# DEM Conditioning: Cross-Region Test on Cambridge Bay (Phase 2, Candidate 1)

The decisive test for `dem_unet/03`. In-region validation showed DEM
conditioning improves ZNCC (0.2344 -> 0.2866, paired mean diff +0.0522,
95% CI [+0.0375, +0.0665], better on 69% of 255 patches) while degrading
the spatial spectrum (log-PSD RMSE +0.3794, worse on 96% of patches).

**Why that result is not yet interpretable.** The DEM is static and
informative *within the same spatial blocks the model trained on*. A model
that learned "read the DEM, largely ignore the SAR" produces exactly that
table. In-region validation cannot distinguish a genuine terrain prior from
a within-region shortcut. Cambridge Bay can: the model has never seen it,
and its ArcticDEM is new to the model too.

**The two outcomes are both publishable.**
- Cross-region ZNCC meaningfully above `09`'s 0.0051 -> the DEM taught the
  model something transferable about terrain. A real contribution.
- Cross-region ZNCC still ~0 -> the in-region gain was a shortcut, and the
  physical story stands: C-band backscatter carries cm-scale surface texture
  that does not determine meter-scale relief, and supplying the relief
  directly does not manufacture the missing fine scales.

**Baseline to beat (`pcrtc/13`, `09`'s checkpoint, same 400 patches):**
ZNCC 0.0051, with `pred_std` ~4x `gt_std` -- the no-DEM model hallucinated
Tuktoyaktuk-style roughness onto genuinely flatter terrain. Watch
`pred_std_val` here as closely as ZNCC: the DEM could plausibly *fix* that
over-confidence by telling the model the terrain is flat, which would show
up as `pred_std` falling toward `gt_std` even if ZNCC barely moves. That
would itself be a finding.

**Confound inherited from `pcrtc/13`, unchanged and not fixable here**:
Cambridge Bay's nearest Sentinel-1 imagery is ~402-409 days after the LiDAR
survey and in a different season (post-thaw summer vs. the April frozen-season
survey). Any result reflects region generalization *and* that season/date
confound. It applies identically to the `09` baseline, so the *comparison*
between the two models is clean even though neither absolute number is.

**This notebook does not execute automatically. Run cells top to bottom.**
Part A (DEM extraction) is CPU/network work; Part B (inference) needs the GPU.

## Setup

In [ ]:
import os
import sys
import json
import random
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import rasterio
from rasterio.warp import transform_bounds, reproject, Resampling
from rasterio.merge import merge as rio_merge
import matplotlib.pyplot as plt

import pystac_client
from shapely.geometry import box as shapely_box, shape
from shapely.ops import unary_union
from concurrent.futures import ThreadPoolExecutor, as_completed

assert torch.cuda.is_available(), 'CUDA is required for Part B. Run this notebook on the GPU environment.'
DEVICE = torch.device('cuda')
torch.backends.cudnn.benchmark = True
print('GPU:', torch.cuda.get_device_name(0))

## Paths and configuration

Cambridge Bay data paths come from `pcrtc/13`; the checkpoint and model
hyperparameters come from `dem_unet/03`. Every value that affects the
comparison (`CONTEXT_K`, `TARGET_HW`, `TIMESTEPS`, `NOISE_SCHEDULE`,
`ATTENTION_VARIANT`, `N_TEST_PATCHES`, `TEST_SAMPLE_SEED`) is identical to
`13`, so the only difference between this run and the baseline is the
DEM branch.

In [ ]:
WORKING_REPO = Path('/cs/student/project_msc/2025/aibh/jiayiche')
TESSA_REPO = WORKING_REPO / 'tessa_baseline'

# Cambridge Bay data -- same paths pcrtc/13 used
LIDAR_DIR = WORKING_REPO / 'input_data' / 'lidar_patches_cambridge_extracted' / 'lidar_patches_cambridge'
S1_DIR = WORKING_REPO / 'input_data' / 's1_patches_cambridge_pcrtc'
DEM_DIR = WORKING_REPO / 'input_data' / 'dem_patches_cambridge'   # produced by Part A below
DEM_DIR.mkdir(parents=True, exist_ok=True)

CHECKPOINT_DIR = WORKING_REPO / 'checkpoints'
OUTPUT_DIR = WORKING_REPO / 's1_training_outputs'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# The DEM-conditioned Tuktoyaktuk checkpoint from dem_unet/03 -- frozen, eval only
CHECKPOINT_NAME = 's1_tuk_pcrtc_dem_realattrs_spatialsplit_unet_best.pth'
METRICS_FILENAME = 's1_pcrtc_dem_cambridgebay_crossregion_metrics.json'
BASELINE_METRICS_FILENAME = 's1_pcrtc_cambridgebay_crossregion_metrics.json'  # pcrtc/13, 09's checkpoint

PGC_STAC_URL = 'https://stac.pgc.umn.edu/api/v1/'
DEM_COLLECTION = 'arcticdem-mosaics-v4.1-10m'

# All identical to pcrtc/13 -- do not change, the comparison depends on it
CONTEXT_K = 3
TARGET_HW = (256, 256)
BATCH_SIZE = 8
TIMESTEPS = 1000
NOISE_SCHEDULE = 'linear'
ATTENTION_VARIANT = 'default'
N_TEST_PATCHES = 400
TEST_SAMPLE_SEED = 42
CAMBRIDGE_BAY_SURVEY_DATE = __import__('datetime').date(2024, 4, 18)

# From dem_unet/03
DEM_FEAT_CH = 16
SEED = 42

print('LIDAR_DIR:', LIDAR_DIR)
print('DEM_DIR:  ', DEM_DIR)
print('Checkpoint:', CHECKPOINT_DIR / CHECKPOINT_NAME)
assert (CHECKPOINT_DIR / CHECKPOINT_NAME).exists(), 'DEM checkpoint not found -- run dem_unet/03 to completion first.'

# Part A -- ArcticDEM extraction over the Cambridge Bay patches

Mechanically identical to `dem_unet/02`, pointed at Cambridge Bay's LiDAR
patches instead of Tuktoyaktuk's. ArcticDEM covers the whole Arctic, so
unlike the Copernicus GLO-30 attempt this should find genuine coverage --
but the same verification steps run anyway rather than assuming it.

In [ ]:
def aoi_from_lidar_patches(patches_dir, max_files=300, workers=8):
    from rasterio.warp import transform_geom
    paths = sorted(patches_dir.glob('lidar_patch_*.tif'))
    if len(paths) > max_files:
        stride = len(paths) / max_files
        paths = [paths[int(i * stride)] for i in range(max_files)]
    def read_bounds(path):
        with rasterio.open(path) as src:
            return src.crs, src.bounds
    results = []
    with ThreadPoolExecutor(max_workers=workers) as pool:
        futures = [pool.submit(read_bounds, p) for p in paths]
        for future in as_completed(futures):
            results.append(future.result())
    crs = results[0][0]
    native = unary_union([shapely_box(*bounds) for _, bounds in results])
    geojson = transform_geom(crs, 'EPSG:4326', native.__geo_interface__)
    return shape(geojson).buffer(0)

aoi = aoi_from_lidar_patches(LIDAR_DIR)
aoi_ll = aoi.convex_hull
print(f'Cambridge Bay AOI bounds (WGS84): {aoi_ll.bounds}')

catalog = pystac_client.Client.open(PGC_STAC_URL)
dem_items = list(catalog.search(collections=[DEM_COLLECTION], bbox=list(aoi_ll.bounds)).items())
print(f'DEM tiles found covering the AOI: {len(dem_items)}')
assert dem_items, f'No {DEM_COLLECTION} coverage found for Cambridge Bay -- stop and investigate before continuing.'
for item in dem_items:
    print(' ', item.id)

## Mosaic the tiles

Cambridge Bay is likely to span more than one ArcticDEM tile (Tuktoyaktuk
needed only one). `rio_merge` handles that; the real `-9999.0` nodata is
read from the source and passed through explicitly so voids are excluded
rather than silently averaged in as valid elevations.

In [ ]:
dem_srcs = [rasterio.open(item.assets['dem'].href) for item in dem_items]
dem_nodata = dem_srcs[0].nodata
print(f'Source tile nodata value: {dem_nodata}')
dem_mosaic, dem_transform = rio_merge(dem_srcs, nodata=dem_nodata)
dem_crs = dem_srcs[0].crs
for src in dem_srcs:
    src.close()

valid = dem_mosaic[0][dem_mosaic[0] != dem_nodata]
print(f'Mosaic shape: {dem_mosaic.shape}, CRS: {dem_crs}')
print(f'Fraction valid (not nodata): {float((dem_mosaic[0] != dem_nodata).mean()):.4f}')
assert valid.size, 'Mosaic is entirely nodata -- do not continue.'
print(f'Elevation range in mosaic (valid pixels only): {float(valid.min()):.1f} to {float(valid.max()):.1f} m')

## Per-patch extraction onto each Cambridge Bay LiDAR patch grid

Destination grid comes from each individual patch bounds/CRS/shape, never
from the mosaic -- the same rule this project has had to relearn three
times now.

In [ ]:
lidar_ids = {p.stem.split('_')[-1] for p in LIDAR_DIR.glob('lidar_patch_*.tif')}
s1_ids = {p.name.split('_')[-1] for p in S1_DIR.glob('s1_patch_*') if p.is_dir()}
paired_ids = sorted(lidar_ids & s1_ids)
print(f'Paired LiDAR/S1 Cambridge Bay patches: {len(paired_ids)}')

matched, skipped_nan = 0, []
for pid in paired_ids:
    with rasterio.open(LIDAR_DIR / f'lidar_patch_{pid}.tif') as lsrc:
        dst_crs, dst_transform = lsrc.crs, lsrc.transform
        out_h, out_w = lsrc.height, lsrc.width

    dst = np.full((out_h, out_w), np.nan, dtype=np.float32)
    reproject(
        source=dem_mosaic[0], destination=dst,
        src_transform=dem_transform, src_crs=dem_crs,
        dst_transform=dst_transform, dst_crs=dst_crs,
        resampling=Resampling.bilinear,
        src_nodata=dem_nodata, dst_nodata=np.nan,
    )

    finite_frac = float(np.isfinite(dst).mean())
    if finite_frac < 0.99:
        skipped_nan.append((pid, finite_frac))
        continue

    meta = {'driver': 'GTiff', 'count': 1, 'height': out_h, 'width': out_w,
            'dtype': 'float32', 'crs': dst_crs, 'transform': dst_transform, 'nodata': np.nan}
    with rasterio.open(DEM_DIR / f'dem_patch_{pid}.tif', 'w', **meta) as dst_file:
        dst_file.write(dst, 1)
    matched += 1

print(f'DEM patches written: {matched} / {len(paired_ids)}')
print(f'Skipped (finite_frac < 0.99): {len(skipped_nan)}')
if skipped_nan:
    print('First few skipped:', skipped_nan[:5])

## Verification -- coverage and plausible values before trusting the data

In [ ]:
dem_paths = sorted(DEM_DIR.glob('dem_patch_*.tif'))
print(f'Total Cambridge Bay DEM patches on disk: {len(dem_paths)}')
assert dem_paths, 'No DEM patches were written -- stop here.'

sample_stats = []
for p in dem_paths[:50]:
    with rasterio.open(p) as src:
        arr = src.read(1)
    sample_stats.append({'finite_frac': float(np.isfinite(arr).mean()),
                         'min_m': float(np.nanmin(arr)), 'max_m': float(np.nanmax(arr)),
                         'std_m': float(np.nanstd(arr))})

print(f"Mean finite_frac (first 50): {np.mean([s['finite_frac'] for s in sample_stats]):.4f}")
print(f"Elevation range across sample: {min(s['min_m'] for s in sample_stats):.1f} to {max(s['max_m'] for s in sample_stats):.1f} m")
print(f"Mean within-patch DEM std: {np.mean([s['std_m'] for s in sample_stats]):.3f} m")
print('\n(Tuktoyaktuk reference: elevations ~-6.7 to -6.4 m over the 50-patch sample.')
print(' ArcticDEM heights are WGS84-ellipsoidal, so small negative values over')
print(' near-sea-level Arctic terrain are expected, not an error. What matters for')
print(' conditioning is the *within-patch* variation, since the DEM is demeaned')
print(' per patch before it reaches the model.)')

# Part B -- cross-region inference with the DEM-conditioned model

Model definition copied verbatim from `dem_unet/03`. It has to match exactly:
loading `03`'s checkpoint into a differently-shaped architecture would either
throw or, worse, silently load a subset of the weights.

In [ ]:
sys.path.insert(0, str(TESSA_REPO))
from src.model.unet import ConditionalUNet
from src.model.blocks import DoubleConv
from src.diffusion.utils import timestep_embedding
from src.diffusion.scheduler import LinearDiffusionScheduler, CosineDiffusionScheduler
from src.diffusion.sampling import p_sample_loop_ddim
from src.utils.recon_metrics import rmse, bias, sigma_error, normal_angle_error, average_jsd_multiscale, log_psd_rmse, zncc

def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

seed_everything(SEED)


class DEMConditionalUNet(ConditionalUNet):
    """ConditionalUNet + one additional, independent DEM conditioning branch."""

    def __init__(self, *args, dem_feat_ch=16, **kwargs):
        super().__init__(*args, **kwargs)
        self.dem_feat_ch = dem_feat_ch
        self.dem_encoder = nn.Sequential(
            nn.Conv2d(1, 16, 3, padding=1), nn.SiLU(),
            nn.Conv2d(16, dem_feat_ch, 3, padding=1), nn.SiLU(),
        )
        in_channels = kwargs.get('in_channels', 1)
        cond_channels = kwargs.get('cond_channels', 24)
        self.input_conv = DoubleConv(in_channels + cond_channels + dem_feat_ch, self.base_channels, self.embed_dim)

    def forward(self, x, cond_img, attrs, dem, t):
        B = x.size(0)
        k = self.cond_k
        A_per = self.attr_dim_per

        t_emb = self.time_mlp(timestep_embedding(t, self.embed_dim))

        attrs_kA = attrs.view(B, k, A_per)
        fused_cond = self.asap(cond_img, attrs_kA, k=k)
        dem_feat = self.dem_encoder(dem)

        x = torch.cat([x, fused_cond, dem_feat], dim=1)

        skips = []
        x = self.input_conv(x, t_emb)
        for down in self.downs:
            skips.append(x)
            x = down(x, t_emb)

        x = self.bottleneck_conv(x, t_emb)

        for up in self.ups:
            skip = skips.pop()
            x = up(x, skip, t_emb)

        return self.output_conv(x)


class DemBoundModel(nn.Module):
    """p_sample_loop_ddim calls model(x, cond, attrs, t) with a fixed 4-arg
    signature, so the DEM is bound in rather than passed through. This reuses
    Tessa's exact DDIM implementation unchanged -- important, since a
    reimplemented sampler could differ subtly from the one 09 and 13 were
    evaluated with, contaminating the comparison."""

    def __init__(self, inner, dem):
        super().__init__()
        self.inner = inner
        self.dem = dem

    def forward(self, x, cond, attrs, t):
        return self.inner(x, cond, attrs, self.dem, t)

## Dataset adapter

`pcrtc/13`'s `CambridgeBayS1Dataset` plus the DEM loading from `dem_unet/03`.
The DEM is demeaned per patch, exactly as in training -- this matters more
here than it did in-region: it strips absolute elevation, so the model cannot
key on Cambridge Bay simply sitting at a different height than Tuktoyaktuk.
Only within-patch relief reaches the network.

In [ ]:
def build_real_attrs(s1_path, times, context_k, survey_date):
    attrs_path = s1_path / 'attrs.json'
    attrs_list = json.load(open(attrs_path)) if attrs_path.exists() else []
    vecs = []
    for time_path in times:
        idx = int(time_path.stem[1:])
        a = attrs_list[idx] if idx < len(attrs_list) else {}
        if a.get('acquisition_date'):
            import datetime as dt
            acq_date = dt.date.fromisoformat(a['acquisition_date'])
            age_norm = (acq_date - survey_date).days / 30.0
        else:
            age_norm = 0.0
        orbit_dir = 1.0 if a.get('orbit_direction') == 'ASCENDING' else 0.0
        rel_orbit = (a.get('relative_orbit_number') or 0) / 175.0
        vecs.append([age_norm, orbit_dir, rel_orbit, 0.0, 0.0, 0.0, 0.0, 0.0])
    return torch.tensor(vecs, dtype=torch.float32).flatten()


class CambridgeBayS1DemDataset(Dataset):
    def __init__(self, s1_dir, lidar_dir, dem_dir, patch_ids, context_k=3, target_hw=(256, 256), survey_date=None):
        self.s1_dir = Path(s1_dir)
        self.lidar_dir = Path(lidar_dir)
        self.dem_dir = Path(dem_dir)
        self.patch_ids = list(patch_ids)
        self.context_k = context_k
        self.target_hw = target_hw
        self.survey_date = survey_date

    def __len__(self):
        return len(self.patch_ids)

    def __getitem__(self, index):
        patch_id = self.patch_ids[index]
        lidar_path = self.lidar_dir / f'lidar_patch_{patch_id}.tif'
        s1_path = self.s1_dir / f's1_patch_{patch_id}'
        dem_path = self.dem_dir / f'dem_patch_{patch_id}.tif'

        with rasterio.open(lidar_path) as src:
            raw = src.read().astype(np.float32)
        target = raw[0]
        mask = (raw[1] > 0.5) if raw.shape[0] > 1 else np.isfinite(target)
        target = np.nan_to_num(target, nan=0.0, posinf=0.0, neginf=0.0)
        valid_count = max(1, int(mask.sum()))
        patch_mean = float(target[mask].sum() / valid_count)
        target = (target - patch_mean) * mask

        times = sorted(s1_path.glob('t*.tif'))[:self.context_k]
        if len(times) < self.context_k:
            raise RuntimeError(f'{s1_path} has fewer than {self.context_k} Sentinel-1 times')
        views = []
        for time_path in times:
            with rasterio.open(time_path) as src:
                sar = src.read()[:2].astype(np.float32)
            sar = np.nan_to_num(sar, nan=0.0, posinf=0.0, neginf=0.0)
            sar = np.maximum(sar, 1e-12)
            sar = 10.0 * np.log10(sar)
            sar_tensor = torch.from_numpy(sar).unsqueeze(0)
            sar_tensor = F.interpolate(sar_tensor, size=self.target_hw, mode='bilinear', align_corners=False).squeeze(0)
            sar_tensor = sar_tensor.repeat(2, 1, 1)
            views.append(sar_tensor)
        condition = torch.cat(views, dim=0)
        attrs = build_real_attrs(s1_path, times, self.context_k, self.survey_date)

        with rasterio.open(dem_path) as src:
            dem_arr = src.read(1).astype(np.float32)
        dem_arr = np.nan_to_num(dem_arr, nan=0.0, posinf=0.0, neginf=0.0)
        dem_arr = dem_arr - dem_arr.mean()  # demean per-patch, same convention as training
        dem_tensor = torch.from_numpy(dem_arr).unsqueeze(0)
        if dem_tensor.shape[-2:] != torch.Size(self.target_hw):
            dem_tensor = F.interpolate(dem_tensor.unsqueeze(0), size=self.target_hw, mode='bilinear', align_corners=False).squeeze(0)

        return {
            'lidar': torch.from_numpy(target).unsqueeze(0).float(), 'mask': torch.from_numpy(mask),
            's1': condition.float(), 'attrs': attrs, 'dem': dem_tensor.float(),
            'patch_mean': torch.tensor(patch_mean), 'patch_id': patch_id,
        }

## Test set -- the same 400 patches `pcrtc/13` used

`all_test_ids` and the seeded 400-patch sample are built exactly as in `13`,
so the comparison is paired patch-for-patch rather than sample-against-sample.
Any patch that failed DEM extraction is dropped and reported; the final
comparison is computed on the intersection with `13`'s saved rows, so a few
dropped patches weaken nothing.

In [ ]:
all_test_ids = sorted(lidar_ids & s1_ids)   # same sets Part A built
assert all_test_ids, 'No paired Cambridge Bay Sentinel-1/LiDAR patches found.'
print(f'Total matched Cambridge Bay patches available: {len(all_test_ids)}')

test_ids = random.Random(TEST_SAMPLE_SEED).sample(all_test_ids, min(N_TEST_PATCHES, len(all_test_ids)))
print(f'Sampled {len(test_ids)} patches (seed={TEST_SAMPLE_SEED}) -- identical selection to pcrtc/13')

have_dem = {p.stem.split('_')[-1] for p in DEM_DIR.glob('dem_patch_*.tif')}
missing_dem = [pid for pid in test_ids if pid not in have_dem]
test_ids = [pid for pid in test_ids if pid in have_dem]
print(f'Dropped for missing DEM: {len(missing_dem)}  ->  {len(test_ids)} patches will be evaluated')
if missing_dem:
    print('First few dropped:', missing_dem[:5])
assert len(test_ids) > 300, 'Too many patches lost to missing DEM -- investigate Part A before trusting any comparison.'

test_dataset = CambridgeBayS1DemDataset(S1_DIR, LIDAR_DIR, DEM_DIR, test_ids, CONTEXT_K, TARGET_HW,
                                        survey_date=CAMBRIDGE_BAY_SURVEY_DATE)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=True)

## Load the DEM-conditioned Tuktoyaktuk checkpoint (frozen, no further training)

In [ ]:
model = DEMConditionalUNet(
    in_channels=1, cond_channels=4 * CONTEXT_K, attr_dim=8 * CONTEXT_K, base_channels=128,
    embed_dim=256, unet_depth=4, attention_variant=ATTENTION_VARIANT, cond_k=CONTEXT_K,
    dem_feat_ch=DEM_FEAT_CH,
).to(DEVICE)
scheduler = LinearDiffusionScheduler(TIMESTEPS, device=DEVICE) if NOISE_SCHEDULE == 'linear' else CosineDiffusionScheduler(TIMESTEPS, device=DEVICE)

checkpoint = torch.load(CHECKPOINT_DIR / CHECKPOINT_NAME, map_location=DEVICE)
model.load_state_dict(checkpoint['model_state_dict'])   # strict=True: any shape mismatch fails loudly here
model.eval()
print(f'Loaded checkpoint from epoch {checkpoint["epoch"]}, val_loss={checkpoint["val_loss"]:.6f}')
print('Parameters:', f'{sum(p.numel() for p in model.parameters()):,}')

# The check that actually proves the DEM branch exists -- parameter count cannot,
# since the branch is only ~21K params on a ~104.6M model
assert model.input_conv.conv1.in_channels == 29, 'DEM branch not wired in.'
print('DEM branch confirmed: input_conv.conv1.in_channels =', model.input_conv.conv1.in_channels)
if checkpoint['epoch'] < 10:
    print(f'\nWARNING: checkpoint is from epoch {checkpoint["epoch"]}. 09 best-val landed at epoch 93 --')
    print('a very early checkpoint means the training run was cut short and this test is meaningless.')

## Run inference

Same DDIM sampler, same metric suite, same batch size as `pcrtc/13`. Expect
roughly the same wall-clock time `13` took for 400 patches.

In [ ]:
sampler = p_sample_loop_ddim
metric_rows = []
example_patches = []
N_EXAMPLES = 6
with torch.no_grad():
    for batch in test_loader:
        target = batch['lidar'].to(DEVICE)
        condition = batch['s1'].to(DEVICE)
        attrs = batch['attrs'].to(DEVICE)
        dem = batch['dem'].to(DEVICE)
        mask = batch['mask'].to(DEVICE).bool()
        bound = DemBoundModel(model, dem)
        prediction = sampler(bound, scheduler, target.shape, condition, attrs, DEVICE)
        means = batch['patch_mean'].to(DEVICE).view(-1, 1, 1, 1)
        gt_absolute = target + means
        pred_absolute = prediction + means
        for i, patch_id in enumerate(batch['patch_id']):
            gt_i, pred_i, mask_i = gt_absolute[i], pred_absolute[i], mask[i]
            gt_valid = gt_i.squeeze()[mask_i].cpu().numpy()
            pred_valid = pred_i.squeeze()[mask_i].cpu().numpy()
            metric_rows.append({
                'patch_id': patch_id,
                'rmse_m': float(rmse(gt_i, pred_i, mask_i).item()),
                'bias_m': float(bias(gt_i, pred_i, mask_i).item()),
                'sigma_error_pct': float(sigma_error(gt_i, pred_i, mask_i).item()),
                'normal_angle_error_deg': float(normal_angle_error(gt_i, pred_i, mask_i, pixel_size=1.0, degrees=True).item()),
                'jsd': float(average_jsd_multiscale(gt_i, pred_i, pixel_size=1.0, mask=mask_i).item()),
                'psd_rmse': float(log_psd_rmse(gt_i, pred_i, pixel_size=1.0, mask=mask_i).item()),
                'zncc': float(zncc(gt_i, pred_i, mask_i).item()),
                'gt_std_val': float(gt_valid.std()) if gt_valid.size > 0 else float('nan'),
                'pred_std_val': float(pred_valid.std()) if pred_valid.size > 0 else float('nan'),
            })
            if len(example_patches) < N_EXAMPLES:
                example_patches.append({
                    'patch_id': patch_id, 'gt': gt_i.squeeze().cpu().numpy(),
                    'pred': pred_i.squeeze().cpu().numpy(), 'mask': mask_i.squeeze().cpu().numpy(),
                    'dem': dem[i].squeeze().cpu().numpy(), 's1_condition': condition[i].cpu().numpy(),
                })

metrics_path = OUTPUT_DIR / METRICS_FILENAME
with metrics_path.open('w') as handle:
    json.dump(metric_rows, handle, indent=2)
print('Saved:', metrics_path)
print('Mean metrics:', {k: float(np.nanmean([r[k] for r in metric_rows])) for k in metric_rows[0] if k != 'patch_id'})

# Part C -- the comparison that answers the question

Paired against `pcrtc/13`'s per-patch rows (same patches, same sampler, same
metrics; `09`'s checkpoint instead of `03`'s). Bootstrap CIs rather than a
bare difference of means, matching how the in-region gain was tested.

**Reading the output:**
- `zncc` CI entirely above zero -> the DEM transfers. Real contribution.
- `zncc` CI spanning zero -> the in-region gain did not transfer; it was a
  within-region shortcut.
- `pred_std_val` moving *down* toward `gt_std_val` -> the DEM curbed the
  hallucinated-roughness failure mode, which is a separate finding worth
  reporting whatever ZNCC does.

In [ ]:
baseline_path = OUTPUT_DIR / BASELINE_METRICS_FILENAME
assert baseline_path.exists(), f'Baseline metrics not found at {baseline_path} -- run pcrtc/13 first.'
baseline = {r['patch_id']: r for r in json.load(baseline_path.open())}
dem_rows = {r['patch_id']: r for r in metric_rows}

ids = sorted(set(baseline) & set(dem_rows))
print(f'paired patches: {len(ids)}  (13 baseline={len(baseline)}, DEM={len(dem_rows)})\n')

METRICS = ['zncc', 'rmse_m', 'psd_rmse', 'jsd', 'sigma_error_pct', 'normal_angle_error_deg',
           'gt_std_val', 'pred_std_val']
HIGHER_IS_BETTER = {'zncc'}

print(f"{'metric':<24}{'13 (no DEM)':>14}{'+ DEM':>12}")
for m in METRICS:
    b = float(np.nanmean([baseline[i][m] for i in ids]))
    d = float(np.nanmean([dem_rows[i][m] for i in ids]))
    print(f'{m:<24}{b:>14.4f}{d:>12.4f}')

print()
rng = np.random.default_rng(42)
for m in ['zncc', 'rmse_m', 'psd_rmse', 'jsd', 'pred_std_val']:
    diff = np.array([dem_rows[i][m] - baseline[i][m] for i in ids])
    diff = diff[np.isfinite(diff)]
    boot = np.array([rng.choice(diff, diff.size, replace=True).mean() for _ in range(10000)])
    lo, hi = np.percentile(boot, [2.5, 97.5])
    direction = 'better' if (m in HIGHER_IS_BETTER) == (diff.mean() > 0) else 'worse'
    sig = 'CI excludes 0' if (lo > 0) == (hi > 0) else 'CI SPANS 0 -- not distinguishable'
    print(f'{m:<16} mean diff {diff.mean():+.4f}  95% CI [{lo:+.4f}, {hi:+.4f}]  '
          f'DEM higher on {float((diff > 0).mean()):.0%}  ({direction}, {sig})')

print('\nIn-region reference (dem_unet/03, 255 patches): zncc +0.0522 [+0.0375, +0.0665], 69% of patches.')
print('If the cross-region zncc CI spans zero while the in-region one did not,')
print('the in-region gain was a within-region shortcut, not a transferable terrain prior.')

## GT-vs-pred residual std scatter -- did the DEM curb the hallucinated roughness?

In [ ]:
import pandas as pd

df = pd.DataFrame(metric_rows)
x, y = df['gt_std_val'], df['pred_std_val']

plt.figure(figsize=(6, 6))
plt.scatter(x, y, alpha=0.5)
mn, mx = float(np.nanmin([x.min(), y.min()])), float(np.nanmax([x.max(), y.max()]))
plt.plot([mn, mx], [mn, mx], 'r--', label='1:1')
ok = np.isfinite(x) & np.isfinite(y)
z = np.poly1d(np.polyfit(x[ok], y[ok], 1))
plt.plot([mn, mx], z([mn, mx]), 'b-', label='Best fit')
r2 = np.corrcoef(x[ok], y[ok])[0, 1] ** 2
plt.text(0.05, 0.95, f'R\u00b2={r2:.3f}', transform=plt.gca().transAxes, va='top',
         bbox=dict(facecolor='white', alpha=0.7))
plt.xlabel('Ground truth')
plt.ylabel('Prediction')
plt.title('Cambridge Bay Cross-Region, DEM-Conditioned\nLiDAR Residual Standard Deviation')
plt.legend()
plt.gca().set_aspect('equal', adjustable='box')
out = OUTPUT_DIR / 's1_pcrtc_dem_cambridgebay_crossregion_gt_pred_std_scatter.png'
plt.savefig(out, dpi=150, bbox_inches='tight')
print('Saved:', out)
print('pcrtc/13 (no DEM) reference: points sat far ABOVE the 1:1 line -- pred_std ~4x gt_std.')
print('Points closer to 1:1 here means the DEM curbed that over-confidence.')
plt.show()

## Reconstruction grid -- S1 / DEM / GT / Pred / Error

The DEM row is the addition to `13`'s version of this figure. It is the
qualitative counterpart to the numbers: if predictions visibly track the DEM
row rather than the S1 rows, that is the shortcut showing up directly.

In [ ]:
n_show = min(6, len(example_patches))
centered_gt = [p['gt'] - p['gt'][p['mask']].mean() for p in example_patches[:n_show]]
centered_pred = [p['pred'] - p['pred'][p['mask']].mean() for p in example_patches[:n_show]]

all_vals = np.concatenate([np.stack(centered_gt).ravel(), np.stack(centered_pred).ravel()])
vmax = float(np.quantile(np.abs(all_vals), 0.995))
err_max = float(np.quantile(np.abs(np.stack(centered_pred) - np.stack(centered_gt)).ravel(), 0.995))

row_titles = ['S1 VV view 1', 'ArcticDEM (demeaned)', 'GT LiDAR (centered)', 'Pred LiDAR (centered)', 'Error (pred - GT)']
fig, axes = plt.subplots(len(row_titles), n_show, figsize=(3.2 * n_show + 2, 3.2 * len(row_titles)), squeeze=False)

for col in range(n_show):
    p = example_patches[col]
    axes[0, col].set_title(f"Patch {p['patch_id']}", fontsize=12, fontweight='bold')
    axes[0, col].imshow(p['s1_condition'][0], cmap='gray')
    axes[1, col].imshow(p['dem'], cmap='terrain')
    axes[2, col].imshow(centered_gt[col], cmap='RdBu_r', vmin=-vmax, vmax=vmax)
    axes[3, col].imshow(centered_pred[col], cmap='RdBu_r', vmin=-vmax, vmax=vmax)
    axes[4, col].imshow(centered_pred[col] - centered_gt[col], cmap='RdBu_r', vmin=-err_max, vmax=err_max)
    for row in range(len(row_titles)):
        axes[row, col].axis('off')

for row, title in enumerate(row_titles):
    axes[row, 0].text(-0.08, 0.5, title, transform=axes[row, 0].transAxes, rotation=90,
                      va='center', ha='center', fontsize=12, fontweight='bold')

plt.tight_layout()
out = OUTPUT_DIR / 's1_pcrtc_dem_cambridgebay_crossregion_reconstructions.png'
plt.savefig(out, dpi=150, bbox_inches='tight')
print('Saved:', out)
plt.show()